In [1]:
!pip install requests beautifulsoup4 pandas lxml


### Automated Multi-Page Web Scraping & Extraction Pipeline
This cell implements the complete production pipeline, dynamically traversing paginated structures, handling data types gracefully, cleaning artifacts, and exporting the final data directly to `extracted_books.csv`.

In [16]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time

def run_scraping_pipeline(max_pages=50):
    base_url = "https://books.toscrape.com/"
    current_url = urljoin(base_url, "catalogue/page-1.html")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

    all_extracted_books = []
    page_count = 0

    print("Starting automated web scraping pipeline...")

    while current_url and page_count < max_pages:
        page_count += 1
        print(f"Scraping page {page_count}: {current_url}")

        try:
            response = requests.get(current_url, headers=headers, timeout=20)
            if response.status_code != 200:
                print(f"Skipping page {page_count} due to status code {response.status_code}")
                break
        except Exception as e:
            print(f"Connection error: {e}")
            break

        soup = BeautifulSoup(response.content, "lxml")
        book_pods = soup.select("article.product_pod")

        if not book_pods:
            print("No more products found.")
            break

        for pod in book_pods:
            try:
                # Title
                title = pod.h3.a["title"]

                # Price extraction & cleaning
                raw_price = pod.select_one(".price_color").text.strip()
                clean_price = float(raw_price.replace('Â', '').replace('£', ''))

                # Availability status
                availability = pod.select_one(".availability").text.strip()

                # Rating parsing
                rating_class = pod.select_one("p.star-rating")["class"][1]
                numeric_rating = rating_map.get(rating_class, 0)

                # Book detailed URL
                relative_url = pod.h3.a["href"]
                if "catalogue/" not in relative_url:
                    book_url = urljoin(base_url, f"catalogue/{relative_url}")
                else:
                    book_url = urljoin(base_url, relative_url)

                all_extracted_books.append({
                    "Title": title,
                    "Price_GBP": clean_price,
                    "Numeric_Rating": numeric_rating,
                    "Availability_Status": availability,
                    "URL": book_url
                })
            except Exception as e:
                print(f"Error parsing an item on page {page_count}: {e}")
                continue

        # Locate Next Page
        next_button = soup.select_one("li.next a")
        if next_button:
            relative_next_url = next_button["href"]
            # Handle relative path resolution for pagination dynamically
            if "catalogue/" not in relative_next_url:
                current_url = urljoin(base_url, f"catalogue/{relative_next_url}")
            else:
                current_url = urljoin(base_url, relative_next_url)
            # Courteous scraping delay
            time.sleep(1)
        else:
            current_url = None
            print("Reached the end of pagination.")

    # Exporting results
    final_df = pd.DataFrame(all_extracted_books)
    output_file = "extracted_books.csv"
    final_df.to_csv(output_file, index=False)

    print(f"Pipeline Execution finished successfully!")
    print(f"Total Books extracted: {len(final_df)}")
    print(f"Data saved to {output_file}")
    return final_df

# Run the complete scraping pipeline across all 50 pages
book_df = run_scraping_pipeline(max_pages=50)
display(book_df.head(10))

Starting automated web scraping pipeline...
Scraping page 1: https://books.toscrape.com/catalogue/page-1.html
Scraping page 2: https://books.toscrape.com/catalogue/page-2.html
Scraping page 3: https://books.toscrape.com/catalogue/page-3.html
Scraping page 4: https://books.toscrape.com/catalogue/page-4.html
Scraping page 5: https://books.toscrape.com/catalogue/page-5.html
Scraping page 6: https://books.toscrape.com/catalogue/page-6.html
Scraping page 7: https://books.toscrape.com/catalogue/page-7.html
Scraping page 8: https://books.toscrape.com/catalogue/page-8.html
Scraping page 9: https://books.toscrape.com/catalogue/page-9.html
Scraping page 10: https://books.toscrape.com/catalogue/page-10.html
Scraping page 11: https://books.toscrape.com/catalogue/page-11.html
Scraping page 12: https://books.toscrape.com/catalogue/page-12.html
Scraping page 13: https://books.toscrape.com/catalogue/page-13.html
Scraping page 14: https://books.toscrape.com/catalogue/page-14.html
Scraping page 15: http

,Title,Price_GBP,Numeric_Rating,Availability_Status,URL
0,A Light in the Attic,51.77,3,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,4,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,22.65,1,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,52.15,1,In stock,https://books.toscrape.com/catalogue/the-black...
